# Hatscan — YOLO-pose eğitimi (Colab, ücretsiz T4)

GPU'yu aç: **Runtime → Change runtime type → T4 GPU**, sonra **Runtime → Run all**.

Akış: dataset üret → YOLO-pose fine-tune → kural-tabanlı vs YOLO before/after karne → `best.pt` indir. ~birkaç dakika.

In [ ]:
# 1) Repo (önce GitHub'a push edilmiş olmalı; ya da sol panelden zip yükleyip bu hücreyi atla)
REPO_URL = 'https://github.com/Enesp4rl4k/hatscan.git'
!git clone -q $REPO_URL hatscan && echo 'clone OK'
%cd hatscan

In [ ]:
# 2) Bağımlılıklar
%pip install -q -e . ultralytics supervision

In [ ]:
# 3) Etiketli sentetik dataset (pose, keypoint)
!python scripts/build_dataset.py --out dataset --per-combo 8

In [ ]:
# 4) YOLO-pose fine-tune (T4'te birkaç dakika)
!python scripts/train_yolo.py --data dataset/data.yaml --model yolo11n-pose.pt --epochs 80

In [ ]:
# 5) Kural-tabanlı baseline vs eğitilmiş YOLO → before/after karne
!python scripts/run_yolo_eval.py --weights runs/pose/train/weights/best.pt

In [ ]:
# 6) (opsiyonel) Birkaç val tahminini supervision ile görselleştir
import glob, cv2, supervision as sv
from ultralytics import YOLO
from matplotlib import pyplot as plt
model = YOLO('runs/pose/train/weights/best.pt')
paths = sorted(glob.glob('dataset/images/val/*.png'))[:4]
edge, vtx = sv.EdgeAnnotator(thickness=3), sv.VertexAnnotator(radius=6)
fig, axes = plt.subplots(1, len(paths), figsize=(16, 5))
for ax, p in zip(axes, paths):
    r = model(p, verbose=False)[0]
    kp = sv.KeyPoints.from_ultralytics(r)
    img = cv2.imread(p)
    img = vtx.annotate(edge.annotate(img.copy(), kp), kp)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.show()

In [ ]:
# 7) best.pt indir
from google.colab import files
files.download('runs/pose/train/weights/best.pt')